In [1]:
!wget -q https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv -O pima_diabetes.csv
!wget -q https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv -O titanic.csv
print("Datasets downloaded")

Datasets downloaded


## Task 1: Random Forest — Breast Cancer

In [2]:
"""
Task 1: Random Forest Classifier - Breast Cancer Dataset (sklearn built-in)
Predict whether a tumor is malignant (0) or benign (1).
"""
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix, precision_score,
                              recall_score, f1_score, roc_auc_score, classification_report)

# 1. Load data
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")   # 0 = malignant, 1 = benign

print("Dataset shape:", X.shape)
print("Class balance:\n", y.value_counts())

# 2. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Train Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=None, random_state=42)
rf.fit(X_train, y_train)

# 4. Predictions
y_pred = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]

# 5. Evaluation
acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

print("\n=== Random Forest - Breast Cancer ===")
print(f"Accuracy      : {acc:.4f}")
print(f"Precision     : {prec:.4f}")
print(f"Recall        : {rec:.4f}")
print(f"F1-score      : {f1:.4f}")
print(f"ROC-AUC       : {roc_auc:.4f}")
print("Confusion Matrix:\n", cm)
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=data.target_names))

# Feature importance (bonus insight)
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop 5 important features:\n", importances.head(5))

Dataset shape: (569, 30)
Class balance:
 target
1    357
0    212
Name: count, dtype: int64

=== Random Forest - Breast Cancer ===
Accuracy      : 0.9561
Precision     : 0.9589
Recall        : 0.9722
F1-score      : 0.9655
ROC-AUC       : 0.9931
Confusion Matrix:
 [[39  3]
 [ 2 70]]

Classification Report:
               precision    recall  f1-score   support

   malignant       0.95      0.93      0.94        42
      benign       0.96      0.97      0.97        72

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114


Top 5 important features:
 worst perimeter         0.133100
worst area              0.128052
worst concave points    0.108107
mean concave points     0.094414
worst radius            0.090639
dtype: float64


##Task 2: Logistic Regression — Diabete

In [3]:
"""
Task 2: Logistic Regression - Pima Indians Diabetes Dataset
Predict whether a patient has diabetes (1) or not (0).
Dataset source: https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv
"""
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, precision_score,
                              recall_score, f1_score, roc_auc_score, classification_report)

# 1. Load dataset
columns = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness", "Insulin",
           "BMI", "DiabetesPedigreeFunction", "Age", "Outcome"]
df = pd.read_csv("pima_diabetes.csv", header=None, names=columns)

# 2. Column names already assigned above. Inspect data.
print("Shape:", df.shape)
print(df.head())

# 3. Check for missing / zero values
# In this dataset, 0 is biologically impossible for these columns and indicates missing data
zero_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
print("\nZero-value counts (treated as missing):")
print((df[zero_cols] == 0).sum())

# Replace 0 with NaN then impute with median
df[zero_cols] = df[zero_cols].replace(0, np.nan)
df[zero_cols] = df[zero_cols].fillna(df[zero_cols].median())
print("\nMissing values after imputation:", df.isnull().sum().sum())

# 4. Train/test split (80/20)
X = df.drop("Outcome", axis=1)
y = df["Outcome"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5. Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 6. Train Logistic Regression
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)

# 7. Evaluate
y_pred = log_reg.predict(X_test_scaled)
y_proba = log_reg.predict_proba(X_test_scaled)[:, 1]

acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

print("\n=== Logistic Regression - Diabetes ===")
print(f"Accuracy      : {acc:.4f}")
print(f"Precision     : {prec:.4f}")
print(f"Recall        : {rec:.4f}")
print(f"F1-score      : {f1:.4f}")
print(f"ROC-AUC       : {roc_auc:.4f}")
print("Confusion Matrix:\n", cm)
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# 8. Interpret model coefficients
coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": log_reg.coef_[0]
}).sort_values(by="Coefficient", key=abs, ascending=False)

print("\nModel Coefficients (scaled features, sorted by magnitude):")
print(coef_df)
print("""
Interpretation:
- Positive coefficient -> increases log-odds of diabetes (Outcome=1) as that feature rises.
- Negative coefficient -> decreases log-odds of diabetes.
- Because features are standardized, coefficient magnitude reflects relative importance.
- Glucose typically has the strongest positive effect: higher glucose -> higher diabetes risk.
""")

Shape: (768, 9)
   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  

Zero-value counts (treated as missing):
Glucose            5
BloodPressure     35
SkinThickness    227
Insulin          374
BMI               11
dtype: int64

Missing values after imputation: 0

=== Logistic Regression - Diabetes ===
Accuracy      : 0.7078
Precision     : 0.6000
Recall

## Task 3: XGBoost — Titanic

In [4]:
"""
Task 3: XGBoost Classifier - Titanic Dataset
Predict whether a passenger survived (1) or not (0).
Dataset source: https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv
"""
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix, precision_score,
                              recall_score, f1_score, roc_auc_score, classification_report)

# 1. Load dataset
df = pd.read_csv("titanic.csv")

# 2. Column names already sensible; keep only useful ones
print("Shape:", df.shape)
print(df.columns.tolist())

# 3. Check missing values
print("\nMissing values:\n", df.isnull().sum())

# Handle missing values
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
df.drop(columns=["Cabin", "Ticket", "Name", "PassengerId"], inplace=True)  # too sparse / non-predictive as-is

# Encode categoricals
le_sex = LabelEncoder()
df["Sex"] = le_sex.fit_transform(df["Sex"])          # male=1, female=0
le_emb = LabelEncoder()
df["Embarked"] = le_emb.fit_transform(df["Embarked"])

print("\nMissing values after cleaning:", df.isnull().sum().sum())

# 4. Train/test split (80/20)
X = df.drop("Survived", axis=1)
y = df["Survived"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5. Feature scaling (XGBoost doesn't strictly need it, but included per assignment spec)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 6. Train XGBoost with explicit parameters
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)
xgb_model.fit(X_train_scaled, y_train)

# 7. Evaluate
y_pred = xgb_model.predict(X_test_scaled)
y_proba = xgb_model.predict_proba(X_test_scaled)[:, 1]

acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

print("\n=== XGBoost - Titanic Survival ===")
print(f"Accuracy      : {acc:.4f}")
print(f"Precision     : {prec:.4f}")
print(f"Recall        : {rec:.4f}")
print(f"F1-score      : {f1:.4f}")
print(f"ROC-AUC       : {roc_auc:.4f}")
print("Confusion Matrix:\n", cm)
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# 8. "Interpret model coefficients" -> XGBoost has no linear coefficients,
# so we report feature importances instead (the tree-based equivalent).
importances = pd.Series(xgb_model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nFeature Importances (XGBoost does not have linear coefficients;")
print("gain-based importance is the appropriate equivalent):")
print(importances)

Shape: (891, 12)
['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

Missing values:
 PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Missing values after cleaning: 0

=== XGBoost - Titanic Survival ===
Accuracy      : 0.7989
Precision     : 0.7797
Recall        : 0.6667
F1-score      : 0.7188
ROC-AUC       : 0.8208
Confusion Matrix:
 [[97 13]
 [23 46]]

Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.88      0.84       110
           1       0.78      0.67      0.72        69

    accuracy                           0.80       179
   macro avg       0.79      0.77      0.78       179
weighted avg       0.80      0.80      0.80       179


Feature Importances (XGBoost do

## Task 4: Decision Tree — Diabetes

In [5]:
"""
Task 4: Decision Tree Classifier - Pima Indians Diabetes Dataset
Predict whether a patient has diabetes (0 = No, 1 = Yes).
Compares a full-depth tree with a restricted-depth (max_depth=3) tree.
"""
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score

# 1. Load dataset
columns = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness", "Insulin",
           "BMI", "DiabetesPedigreeFunction", "Age", "Outcome"]
df = pd.read_csv("pima_diabetes.csv", header=None, names=columns)

# 2. Column names assigned above.

# 3. Check for missing / unrealistic zero values and handle them
zero_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
print("Zero-value counts (unrealistic, treated as missing):")
print((df[zero_cols] == 0).sum())

df[zero_cols] = df[zero_cols].replace(0, np.nan)
df[zero_cols] = df[zero_cols].fillna(df[zero_cols].median())
print("\nMissing values after handling:", df.isnull().sum().sum())

# 5. Define X and y
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

# 4. Split 80/20, random_state=42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 6. Train full Decision Tree
dt_full = DecisionTreeClassifier(random_state=42)
dt_full.fit(X_train, y_train)
y_pred_full = dt_full.predict(X_test)

# 7. Evaluate full tree
def evaluate(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    print(f"\n=== {name} ===")
    print(f"Accuracy  : {acc:.4f}")
    print(f"Precision : {prec:.4f}")
    print(f"Recall    : {rec:.4f}")
    print(f"F1-score  : {f1:.4f}")
    print("Confusion Matrix:\n", cm)
    return acc, prec, rec, f1

full_metrics = evaluate("Decision Tree (full depth)", y_test, y_pred_full)

# 8. Restricted-depth tree (max_depth=3)
dt_shallow = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_shallow.fit(X_train, y_train)
y_pred_shallow = dt_shallow.predict(X_test)
shallow_metrics = evaluate("Decision Tree (max_depth=3)", y_test, y_pred_shallow)

# Compare
print("\n=== Comparison: Full vs Restricted (max_depth=3) ===")
comp = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Full Tree": full_metrics,
    "Depth=3 Tree": shallow_metrics
})
print(comp.to_string(index=False))
print("""
Interpretation:
- The full-depth tree tends to overfit the training data (high variance, captures noise),
  which often shows up as good training performance but weaker/less stable test performance.
- The depth-3 tree is simpler and more interpretable, and generally generalizes better on
  unseen data because it is regularized against overfitting.
""")

# Feature importance (from full tree, since it uses all splits)
importances = pd.Series(dt_full.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Feature Importances (full Decision Tree):")
print(importances)

importances_shallow = pd.Series(dt_shallow.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nFeature Importances (max_depth=3 Decision Tree):")
print(importances_shallow)

Zero-value counts (unrealistic, treated as missing):
Glucose            5
BloodPressure     35
SkinThickness    227
Insulin          374
BMI               11
dtype: int64

Missing values after handling: 0

=== Decision Tree (full depth) ===
Accuracy  : 0.7208
Precision : 0.6034
Recall    : 0.6364
F1-score  : 0.6195
Confusion Matrix:
 [[76 23]
 [20 35]]

=== Decision Tree (max_depth=3) ===
Accuracy  : 0.7597
Precision : 0.6800
Recall    : 0.6182
F1-score  : 0.6476
Confusion Matrix:
 [[83 16]
 [21 34]]

=== Comparison: Full vs Restricted (max_depth=3) ===
   Metric  Full Tree  Depth=3 Tree
 Accuracy   0.720779      0.759740
Precision   0.603448      0.680000
   Recall   0.636364      0.618182
 F1-score   0.619469      0.647619

Interpretation:
- The full-depth tree tends to overfit the training data (high variance, captures noise),
  which often shows up as good training performance but weaker/less stable test performance.
- The depth-3 tree is simpler and more interpretable, and general